In [41]:
# TUMOR TAIWAN VS NORMAL TAIWAN
# TAIWAN NORMAL VS TAIWAN TUMOR
# WHITE NORMAL VS WHITE TUMOR
# WHITE TUMOR VS WHITE NORMAL

In [42]:
library(WGCNA)
library(clusterProfiler)
library(ggplot2)

In [43]:
cohort <- "TAIWAN" # "TAIWAN" or "WHITE"
if (cohort == "TAIWAN") {
    config <- list(
        wd = "/home/seba/github_repos/crc_weighted_network/taiwanese_cohort",
        work_tumor = 'rna_tumor',
        work_normal = 'rna_normal'
    )
} else if (cohort == "WHITE") {
    config <- list(
        wd = "/home/seba/github_repos/crc_weighted_network/cohort_white",
        work_tumor = 'rna_tumor_tmm0.4',
        work_normal = 'rna_normal_tmm0.4'
    )
}

In [44]:
setwd(config$wd)

In [45]:
# Intra module preservation on white cohort
datExpr_tumor <- readRDS(paste0(config$work_tumor,"/datExpr_clean.rds"))
datExpr_normal <- readRDS(paste0(config$work_normal,"/datExpr_clean.rds"))

In [46]:
commonGenes <- intersect(colnames(datExpr_tumor), colnames(datExpr_normal))

datExpr_tumor  <- datExpr_tumor[, commonGenes, drop=FALSE]
datExpr_normal <- datExpr_normal[, commonGenes, drop=FALSE]

In [47]:
gene2mod <- read.table(paste0(config$work_tumor,"/module_membership_gene2module.tsv"), header=TRUE, sep="\t", stringsAsFactors=FALSE)
refColors_map <- setNames(gene2mod$module, gene2mod$gene)
refColors <- refColors_map[commonGenes]

In [48]:
multiExpr <- list(
  Ref  = list(data = datExpr_tumor),
  Test = list(data = datExpr_normal)
)

colorList <- list(Ref = refColors)

In [49]:
# Do this only if intra_module_preservation_white_tumor_vs_normal.rds does not exist
if (!file.exists(paste0(cohort, "_intra_module_preservation_tumor_vs_normal.rds"))){
    mp <- modulePreservation(
    multiExpr, colorList,
    referenceNetworks = 1,
    networkType = "signed",
    corFnc = "bicor",     # or "cor" if Pearson
    nPermutations = 200,
    randomSeed = 1,
    verbose = 3
)
saveRDS(mp, file=paste0(cohort, "_intra_module_preservation_tumor_vs_normal.rds"))
} else {
    mp <- readRDS(paste0(cohort, "_intra_module_preservation_tumor_vs_normal.rds"))
}

In [50]:
Ztab  <- mp$preservation$Z$ref.Ref$inColumnsAlsoPresentIn.Test
Ztab <- as.data.frame(Ztab)

In [51]:
out <- data.frame(
  module = rownames(Ztab),
  size = Ztab$moduleSize,
  Zsummary_pres = Ztab$Zsummary.pres,
  Zdensity_pres = Ztab$Zdensity.pres,
  Zconnectivity_pres = Ztab$Zconnectivity.pres,
  stringsAsFactors = FALSE
)

out <- out[order(-out$Zsummary_pres), ]
head(out, 20)

,module,size,Zsummary_pres,Zdensity_pres,Zconnectivity_pres
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
2,blue,1000,69.243411,117.361583,21.125240
37,salmon,337,30.509529,43.337643,17.681415
26,mediumpurple3,86,27.079533,48.776130,5.382936
12,darkturquoise,188,26.444779,41.224514,11.665044
3,brown,958,26.158710,13.456020,38.861399
15,green,605,21.087615,31.478328,10.696901
42,tan,380,18.215295,27.224794,9.205797
31,pink,536,17.269226,23.610635,10.927816
43,turquoise,1000,16.434811,12.033223,20.836399


In [52]:
svg("preserv1.svg")
ggplot(out, aes(x = size, y = Zsummary_pres, color = module, label = module)) +
  geom_point(size = 3) +
  geom_text(vjust = -0.7, size = 3, show.legend = FALSE) +
  geom_hline(yintercept = c(2, 10),
             linetype = "dashed",
             color = "grey40") +
  scale_color_identity() +
  labs(
    x = "Module size (# genes)",
    y = "Zsummary (preservation)",
    title = "Module preservation (reference → test)"
  ) +
  theme_bw()
dev.off()

pdf 
  2

In [53]:
svg("preserv2.svg")
ggplot(out, aes(x = Zdensity_pres, y = Zconnectivity_pres,
                color = module, label = module)) +
  geom_point(size = 3) +
  geom_text(vjust = -0.7, size = 3, show.legend = FALSE) +
  geom_vline(xintercept = 2,
             linetype = "dashed",
             color = "grey40") +
  geom_hline(yintercept = 2,
             linetype = "dashed",
             color = "grey40") +
  scale_color_identity() +
  labs(
    x = "Zdensity (preservation)",
    y = "Zconnectivity (preservation)",
    title = "Preservation components"
  ) +
  theme_bw()
dev.off()

pdf 
  2

In [54]:
gene2mod_normal <- read.table(paste0(config$work_normal,"/module_membership_gene2module.tsv"), header=TRUE, sep="\t", stringsAsFactors=FALSE)
refColors_map <- setNames(gene2mod_normal$module, gene2mod_normal$gene)
refColors <- refColors_map[commonGenes]

In [55]:
multiExpr <- list(
  Ref= list(data = datExpr_normal),
  Test  = list(data = datExpr_tumor)
)

colorList <- list(Ref = refColors)

In [56]:
# Do this only if intra_module_preservation_white_normal_vs_tumor.rds does not exist
if (!file.exists(paste0(cohort, "_intra_module_preservation_normal_vs_tumor.rds"))){
    mp <- modulePreservation(
    multiExpr, colorList,
    referenceNetworks = 1,
    networkType = "signed",
    corFnc = "bicor",     # or "cor" if Pearson
    nPermutations = 200,
    randomSeed = 1,
    verbose = 3
)
saveRDS(mp, file=paste0(cohort, "_intra_module_preservation_normal_vs_tumor.rds"))
} else {
    mp <- readRDS(paste0(cohort, "_intra_module_preservation_normal_vs_tumor.rds"))
}

In [57]:
Ztab  <- mp$preservation$Z$ref.Ref$inColumnsAlsoPresentIn.Test
Ztab <- as.data.frame(Ztab)

In [58]:
out <- data.frame(
  module = rownames(Ztab),
  size = Ztab$moduleSize,
  Zsummary_pres = Ztab$Zsummary.pres,
  Zdensity_pres = Ztab$Zdensity.pres,
  Zconnectivity_pres = Ztab$Zconnectivity.pres,
  stringsAsFactors = FALSE
)

out <- out[order(-out$Zsummary_pres), ]
head(out, 20)

,module,size,Zsummary_pres,Zdensity_pres,Zconnectivity_pres
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
17,tan,450,86.845195,157.8624159,15.827974
3,brown,1000,82.416794,141.6997260,23.133863
13,pink,791,58.855508,85.6658535,32.045163
4,cyan,399,52.065802,89.1746079,14.956996
1,black,812,47.479026,78.5840928,16.373960
11,magenta,710,41.961853,68.5762138,15.347492
6,green,902,38.860011,51.2678974,26.452126
2,blue,1000,31.683686,33.9345971,29.432775
9,grey60,236,31.448586,52.3686976,10.528475


In [59]:
svg(paste0(cohort, "_intra_normal_vs_tumor_1.svg"))
ggplot(out, aes(x = size, y = Zsummary_pres, color = module, label = module)) +
  geom_point(size = 3) +
  geom_text(vjust = -0.7, size = 3, show.legend = FALSE) +
  geom_hline(yintercept = c(2, 10),
             linetype = "dashed",
             color = "grey40") +
  scale_color_identity() +
  labs(
    x = "Module size (# genes)",
    y = "Zsummary (preservation)",
    title = "Module preservation (reference → test)"
  ) +
  theme_bw()
dev.off()

pdf 
  2

In [60]:
svg(paste0(cohort, "_intra_normal_vs_tumor_2.svg"))
ggplot(out, aes(x = Zdensity_pres, y = Zconnectivity_pres,
                color = module, label = module)) +
  geom_point(size = 3) +
  geom_text(vjust = -0.7, size = 3, show.legend = FALSE) +
  geom_vline(xintercept = 2,
             linetype = "dashed",
             color = "grey40") +
  geom_hline(yintercept = 2,
             linetype = "dashed",
             color = "grey40") +
  scale_color_identity() +
  labs(
    x = "Zdensity (preservation)",
    y = "Zconnectivity (preservation)",
    title = "Preservation components"
  ) +
  theme_bw()
dev.off()

pdf 
  2